# Write to Bronze Layer
#### CSV -> PySpark -> Delta Lake -> DuckDB

Steps and actions: 
- Used PERMISSIVE mode with a _corrupt_record column to save malformed rows (according to the transaction schema) without crashing the pipeline.
- Created an index_to_drop field to the Spark schema to catch the unnamed index, then immediately dropped it.
- Used monotonically_increasing_id() to prevent single-node Out-Of-Memory (OOM) crashes.
- Partitioned by _load_date for efficient daily incremental loads and easy reprocessing of specific dates.
- Used Delta Lake format for ACID transactions and incremental data loading.
- Delta Lake with `mode("append")` enables daily incremental loads without overwriting existing data.
- Set `SPARK_LOCAL_IP` and `PYSPARK_SUBMIT_ARGS` environment variables for proper Delta Lake integration in Jupyter (PySpark 3.5.3 + Delta 3.2.0).
- DuckDB reads Delta tables natively using delta_scan().

In [9]:
import os

# Set networking to localhost to avoid WSL networking issues
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name, monotonically_increasing_id, to_date
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DateType, BooleanType
import duckdb as db

In [10]:
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [11]:
transaction_schema = StructType([
    StructField("index_to_drop", IntegerType(), True),
    StructField("product_id", StringType(), False),
    StructField("product_parent", LongType(), True),
    StructField("product_title", StringType(), True),
    StructField("vine", StringType(), True),
    StructField("verified_purchase", StringType(), True),
    StructField("review_headline", StringType(), True),
    StructField("review_body", StringType(), True),
    StructField("review_date", DateType(), True),
    StructField("marketplace_id", LongType(), True),
    StructField("product_category_id", LongType(), True),
    StructField("label", BooleanType(), True),
    
    # catches rows that fail the schema check
    StructField("_corrupt_record", StringType(), True)
])

#### Handle input

In [ ]:
from pyspark.sql.functions import to_date
import shutil

# current working directory
current_dir = os.getcwd()

# project root directory (two levels up from current directory)
project_root = os.path.abspath(os.path.join(current_dir, "../../"))

# output path - use "bronze" instead of "bronze (raw)" to avoid path issues
bronze_output_path = os.path.join(project_root, "data", "bronze")

# input path
raw_csv_dir = os.path.join(project_root, "reviews (copy)")

# Clean up old non-Delta data if needed (first time only)
if os.path.exists(bronze_output_path) and not os.path.exists(os.path.join(bronze_output_path, "_delta_log")):
    shutil.rmtree(bronze_output_path)

print(f"Reading from: {raw_csv_dir}")
print(f"Writing to:   {bronze_output_path}")

# Create Spark df with and test with transaction schema and add metadata columns
base_df = (spark.read 
    .option("header", "true") 
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(transaction_schema) 
    .option("pathGlobFilter", "train-*.csv") 
    .csv(raw_csv_dir)
    .drop("index_to_drop")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", to_date(current_timestamp()))
    .withColumn("_source_file", input_file_name())
    .withColumn("_index", monotonically_increasing_id())
)

Cleaning up old non-Delta data at /home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/data/bronze
Reading from: /home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/reviews (copy)
Writing to:   /home/ronlakeman/MasterInformationStudies/BigDataCourse/big-data-course-2024-projects/BigDataProject/data/bronze


#### Write to data folder

In [15]:
# split the data into 8 partitions 

(base_df
    .repartition(8) 
    .write
    .partitionBy("_load_date")

    # Using Delta Lake format for ACID transactions and incremental data loading
    # Incremental load - appends new data without overwriting
    .mode("append") 
    .format("delta")
    .save(bronze_output_path)
)

26/02/28 19:40:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


#### Database Connection

In [14]:
con = db.connect(os.path.join(project_root, "ProjectData.duckdb"))

# ensure the schema exists first!
con.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

# Read Delta Lake table - DuckDB natively supports Delta format
con.execute(f"""
    CREATE OR REPLACE TABLE bronze.reviews_raw AS 
    SELECT * FROM delta_scan('{bronze_output_path}')
""")

print("Total rows:", con.execute("SELECT COUNT(*) FROM bronze.reviews_raw").fetchone()[0])
print("Corrupt rows:", con.execute("SELECT COUNT(*) FROM bronze.reviews_raw WHERE _corrupt_record IS NOT NULL").fetchone()[0])
print("Unique load dates:", con.execute("SELECT COUNT(DISTINCT _load_date) FROM bronze.reviews_raw").fetchone()[0])

con.close()

Total rows: 9614
Corrupt rows: 1051
Unique load dates: 1
